In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
!pip install -q langchain langgraph langchain-groq langchain-community duckduckgo-search pydantic

import os
import re
import sqlite3
import json
from dataclasses import dataclass
from typing import Optional
from kaggle_secrets import UserSecretsClient

# Load Groq API Key from Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
    print("GROQ_API_KEY loaded successfully.")
except Exception as e:
    print("Ensure GROQ_API_KEY is set in Kaggle Secrets:", e)

GROQ_API_KEY loaded successfully.


In [11]:
DB_PATH = "lab_booking.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # Enable foreign keys
    cur.execute("PRAGMA foreign_keys = ON;")

    # 1. Tests Table
    cur.execute("""
    CREATE TABLE IF NOT EXISTS tests (
        test_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL,
        fasting_required INTEGER NOT NULL
    )
    """)

    # 2. Slots Table
    cur.execute("""
    CREATE TABLE IF NOT EXISTS slots (
        slot_id TEXT PRIMARY KEY,
        test_id TEXT NOT NULL,
        slot_time TEXT NOT NULL,
        is_booked INTEGER DEFAULT 0,
        FOREIGN KEY (test_id) REFERENCES tests(test_id)
    )
    """)

    # 3. Bookings Table
    cur.execute("""
    CREATE TABLE IF NOT EXISTS bookings (
        booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT NOT NULL,
        test_id TEXT NOT NULL,
        slot_id TEXT NOT NULL UNIQUE,
        status TEXT DEFAULT 'CONFIRMED',
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (test_id) REFERENCES tests(test_id),
        FOREIGN KEY (slot_id) REFERENCES slots(slot_id)
    )
    """)

    # Seed 8 Tests across 3 Categories (Pathology, Biochemistry, Radiology)
    tests_data = [
        ("T01", "Complete Blood Count (CBC)", "Pathology", 350.0, 0),
        ("T02", "ESR & Peripheral Smear", "Pathology", 250.0, 0),
        ("T03", "Lipid Profile", "Biochemistry", 800.0, 1),
        ("T04", "HbA1c (Glycated Hemoglobin)", "Biochemistry", 600.0, 0),
        ("T05", "Fasting Blood Glucose", "Biochemistry", 150.0, 1),
        ("T06", "Thyroid Profile (T3, T4, TSH)", "Biochemistry", 750.0, 0),
        ("T07", "Chest X-Ray PA View", "Radiology", 500.0, 0),
        ("T08", "Ultrasound Whole Abdomen", "Radiology", 1400.0, 1),
    ]
    cur.executemany("INSERT OR IGNORE INTO tests VALUES (?, ?, ?, ?, ?)", tests_data)

    # Seed 3+ Slots per test (T02 has all slots booked -> 0 free slots)
    slots_data = []
    times = ["2026-11-20 08:00 AM", "2026-11-20 10:00 AM", "2026-11-20 02:00 PM"]
    for t_id, _, _, _, _ in tests_data:
        for idx, t_str in enumerate(times, 1):
            s_id = f"{t_id}_S{idx}"
            # Make T02 have NO free slots (all is_booked = 1)
            booked_flag = 1 if t_id == "T02" else 0
            slots_data.append((s_id, t_id, t_str, booked_flag))

    cur.executemany("INSERT OR IGNORE INTO slots VALUES (?, ?, ?, ?)", slots_data)
    conn.commit()
    conn.close()

# Run twice to prove idempotency
init_db()
init_db()
print("Database initialized and seeded safely (ran twice without duplicates).")

Database initialized and seeded safely (ran twice without duplicates).


In [14]:
!pip install -q langchain langgraph langchain-groq langchain-community ddgs pydantic tabulate

In [15]:
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.runnables import RunnableConfig
from langchain_community.tools import DuckDuckGoSearchRun
from typing_extensions import Annotated

# --- Requirement 8: Email Redaction ---
def redact_pii(text: str) -> str:
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    return re.sub(email_pattern, "[REDACTED_EMAIL]", text)

# Proof of PII redaction
sample_input = "Hi, book CBC for me, my email is patient.john@example.com"
print("PII Redaction Proof:")
print("  Original :", sample_input)
print("  Redacted :", redact_pii(sample_input))

# --- Requirement 8: Simulated Transient Failure Counter for check_slots ---
CHECK_SLOTS_ATTEMPTS = 0
SIMULATE_FIRST_CALL_FAILURE = True

# --- Requirement 2: Read Tools ---
@tool
def list_tests(category: Optional[str] = None) -> str:
    """List available diagnostic lab tests with their IDs, categories, prices, and fasting requirements.
    Optionally filter by category (Pathology, Biochemistry, Radiology)."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    if category:
        cur.execute(
            "SELECT test_id, name, category, price, fasting_required FROM tests WHERE LOWER(category) = LOWER(?)",
            (category,)
        )
    else:
        cur.execute("SELECT test_id, name, category, price, fasting_required FROM tests")
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return "No tests found for that category."
    
    result = []
    for r in rows:
        fasting = "Yes" if r[4] else "No"
        result.append(f"ID: {r[0]} | Name: {r[1]} | Category: {r[2]} | Price: ₹{r[3]} | Fasting: {fasting}")
    return "\n".join(result)

@tool
def check_slots(test_id: str) -> str:
    """Check available (unbooked) time slots for a specific test_id (e.g., 'T01')."""
    global CHECK_SLOTS_ATTEMPTS, SIMULATE_FIRST_CALL_FAILURE
    CHECK_SLOTS_ATTEMPTS += 1
    
    # Simulate a transient DB lock on the very first call to test retry resilience
    if SIMULATE_FIRST_CALL_FAILURE and CHECK_SLOTS_ATTEMPTS == 1:
        raise RuntimeError("Transient DatabaseTimeoutError: Connection pool busy. Please retry.")

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name FROM tests WHERE test_id = ?", (test_id.upper(),))
    test = cur.fetchone()
    if not test:
        conn.close()
        return f"Error: Test ID '{test_id}' does not exist."

    cur.execute(
        "SELECT slot_id, slot_time FROM slots WHERE test_id = ? AND is_booked = 0",
        (test_id.upper(),)
    )
    slots = cur.fetchall()
    conn.close()
    
    if not slots:
        return f"No free slots available for {test[0]} ({test_id.upper()})."
    
    slot_list = [f"Slot ID: {s[0]} | Time: {s[1]}" for s in slots]
    return f"Available slots for {test[0]} ({test_id.upper()}):\n" + "\n".join(slot_list)

@tool
def my_bookings(config: Annotated[RunnableConfig, InjectedToolArg]) -> str:
    """View all active bookings for the current logged-in user."""
    user_id = config.get("configurable", {}).get("user_id")
    if not user_id:
        return "Error: User identity missing from context."

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("""
        SELECT b.booking_id, t.name, t.price, s.slot_time, b.status, s.slot_id
        FROM bookings b
        JOIN tests t ON b.test_id = t.test_id
        JOIN slots s ON b.slot_id = s.slot_id
        WHERE b.user_id = ? AND b.status = 'CONFIRMED'
    """, (user_id,))
    rows = cur.fetchall()
    conn.close()

    if not rows:
        return "You have no active bookings."
    
    lines = [
        f"Booking #{r[0]} | Test: {r[1]} | Time: {r[3]} (Slot: {r[5]}) | Price: ₹{r[2]} | Status: {r[4]}"
        for r in rows
    ]
    return "\n".join(lines)

# --- Requirement 3: Write Tools ---
@tool
def book_test(test_id: str, slot_id: str, config: Annotated[RunnableConfig, InjectedToolArg]) -> str:
    """Book a diagnostic test for a specific free slot. Requires human approval before execution."""
    user_id = config.get("configurable", {}).get("user_id")
    if not user_id:
        return "Error: User identity missing from context."

    test_id = test_id.upper().strip()
    slot_id = slot_id.upper().strip()

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # 1. Validate test exists
    cur.execute("SELECT name, price FROM tests WHERE test_id = ?", (test_id,))
    test_row = cur.fetchone()
    if not test_row:
        conn.close()
        return f"Booking Failed: Test ID '{test_id}' does not exist."

    # 2. Validate slot exists, belongs to test, and is free
    cur.execute("SELECT test_id, slot_time, is_booked FROM slots WHERE slot_id = ?", (slot_id,))
    slot_row = cur.fetchone()
    if not slot_row:
        conn.close()
        return f"Booking Failed: Slot ID '{slot_id}' does not exist."
    if slot_row[0] != test_id:
        conn.close()
        return f"Booking Failed: Slot '{slot_id}' does not belong to test '{test_id}'."
    if slot_row[2] == 1:
        conn.close()
        return f"Booking Failed: Slot '{slot_id}' is already booked."

    # 3. Validate no duplicate active booking for same user & test
    cur.execute(
        "SELECT booking_id FROM bookings WHERE user_id = ? AND test_id = ? AND status = 'CONFIRMED'",
        (user_id, test_id)
    )
    if cur.fetchone():
        conn.close()
        return f"Booking Failed: You already have an active booking for {test_row[0]} ({test_id})."

    # Execute booking
    cur.execute("UPDATE slots SET is_booked = 1 WHERE slot_id = ?", (slot_id,))
    cur.execute(
        "INSERT INTO bookings (user_id, test_id, slot_id, status) VALUES (?, ?, ?, 'CONFIRMED')",
        (user_id, test_id, slot_id)
    )
    booking_id = cur.lastrowid
    conn.commit()

    # Read receipt directly from the newly inserted database row
    cur.execute("""
        SELECT b.booking_id, b.user_id, t.name, t.price, s.slot_time, b.status, b.created_at
        FROM bookings b
        JOIN tests t ON b.test_id = t.test_id
        JOIN slots s ON b.slot_id = s.slot_id
        WHERE b.booking_id = ?
    """, (booking_id,))
    receipt = cur.fetchone()
    conn.close()

    return (
        f"BOOKING RECEIPT (DB Record #{receipt[0]})\n"
        f"- Patient ID: {receipt[1]}\n"
        f"- Test: {receipt[2]} ({test_id})\n"
        f"- Slot: {receipt[4]} ({slot_id})\n"
        f"- Amount Payable: ₹{receipt[3]}\n"
        f"- Status: {receipt[5]}\n"
        f"- Timestamp: {receipt[6]}"
    )

@tool
def cancel_booking(booking_id: int, config: Annotated[RunnableConfig, InjectedToolArg]) -> str:
    """Cancel an existing booking by booking_id. Requires human approval before execution."""
    user_id = config.get("configurable", {}).get("user_id")
    if not user_id:
        return "Error: User identity missing from context."

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # Validate booking exists and belongs to current user
    cur.execute(
        "SELECT user_id, slot_id, status FROM bookings WHERE booking_id = ?",
        (booking_id,)
    )
    row = cur.fetchone()
    if not row:
        conn.close()
        return f"Cancellation Failed: Booking #{booking_id} does not exist."
    if row[0] != user_id:
        conn.close()
        return f"Cancellation Failed: Booking #{booking_id} does not belong to your account."
    if row[2] != "CONFIRMED":
        conn.close()
        return f"Cancellation Failed: Booking #{booking_id} is already {row[2]}."

    slot_id = row[1]
    cur.execute("UPDATE bookings SET status = 'CANCELLED' WHERE booking_id = ?", (booking_id,))
    cur.execute("UPDATE slots SET is_booked = 0 WHERE slot_id = ?", (slot_id,))
    conn.commit()
    conn.close()

    return f"Booking #{booking_id} has been cancelled and slot {slot_id} is now free."

# --- Requirement 5: Web Search Tool ---
ddg_search = DuckDuckGoSearchRun()

@tool
def web_search_health_info(query: str) -> str:
    """Search the web for general medical/diagnostic test information (e.g., what a test measures, fasting preparation).
    NEVER use this for lab prices, availability, or diagnosing symptoms."""
    try:
        return ddg_search.invoke(f"diagnostic lab test medical information {query}")
    except Exception as e:
        return f"Web search unavailable: {e}"

ALL_TOOLS = [list_tests, check_slots, my_bookings, book_test, cancel_booking, web_search_health_info]

# --- Requirement 4 Proof: Print Tool Schemas seen by the Model ---
print("\n--- Tool Schemas Exposed to LLM (Proving user_id is NOT a model argument) ---")
for t in ALL_TOOLS:
    schema = t.get_input_schema().model_json_schema()
    print(f"Tool: {t.name} -> Arguments: {list(schema.get('properties', {}).keys())}")

PII Redaction Proof:
  Original : Hi, book CBC for me, my email is patient.john@example.com
  Redacted : Hi, book CBC for me, my email is [REDACTED_EMAIL]

--- Tool Schemas Exposed to LLM (Proving user_id is NOT a model argument) ---
Tool: list_tests -> Arguments: ['category']
Tool: check_slots -> Arguments: ['test_id']
Tool: my_bookings -> Arguments: []
Tool: book_test -> Arguments: ['test_id', 'slot_id']
Tool: cancel_booking -> Arguments: ['booking_id']
Tool: web_search_health_info -> Arguments: ['query']


In [16]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = """You are a helpful, accurate Diagnostic Lab Booking Assistant.
Follow these strict rules at all times:
1. DATABASE ONLY FOR PRICES & SLOTS: Always use `list_tests` and `check_slots` to check test prices, test IDs, and time slot availability. NEVER guess or use web search for prices or slots.
2. CONFIRM BEFORE BOOKING OR CANCELLING: Before calling `book_test` or `cancel_booking`, always verify availability/details first and ask the user to confirm the exact test, slot, and price (or booking ID) unless they have already explicitly confirmed those exact details.
3. NEVER DIAGNOSE OR RECOMMEND TESTS FROM SYMPTOMS: If a user describes symptoms (e.g., fatigue, chest pain, fever) or asks what test they need for a medical condition, politely decline to diagnose or recommend tests and advise them to consult a qualified doctor.
4. GENERAL HEALTH INFO: You may use `web_search_health_info` ONLY for general educational questions about what a test measures or general fasting instructions.
5. PRIVACY: Never ask for or repeat sensitive personal information like emails or passwords.
"""

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_with_tools = llm.bind_tools(ALL_TOOLS)
TOOLS_BY_NAME = {t.name: t for t in ALL_TOOLS}
WRITE_TOOLS = {"book_test", "cancel_booking"}

def agent_node(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def tool_execution_node(state: MessagesState, config: RunnableConfig):
    """Executes tool calls with automatic retry logic (Requirement 8)."""
    last_message = state["messages"][-1]
    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_fn = TOOLS_BY_NAME[tool_name]

        # Retry loop (up to 2 attempts) for reliability
        max_retries = 2
        output = None
        for attempt in range(1, max_retries + 1):
            try:
                output = tool_fn.invoke(tool_args, config=config)
                break
            except Exception as e:
                print(f"  [Reliability Log] Tool '{tool_name}' failed on attempt {attempt}: {e}")
                if attempt == max_retries:
                    output = f"Tool error after {max_retries} attempts: {e}"
                else:
                    print(f"  [Reliability Log] Retrying '{tool_name}'...")

        tool_messages.append(
            ToolMessage(
                content=str(output),
                tool_call_id=tool_call["id"],
                name=tool_name
            )
        )
    return {"messages": tool_messages}

def route_after_agent(state: MessagesState):
    last_message = state["messages"][-1]
    if not getattr(last_message, "tool_calls", None):
        return END
    # Check if any tool call requires human approval
    for tc in last_message.tool_calls:
        if tc["name"] in WRITE_TOOLS:
            return "approval_gate"
    return "tools"

# Build Graph
builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("approval_gate", tool_execution_node) # Graph will interrupt BEFORE this node
builder.add_node("tools", tool_execution_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route_after_agent, ["approval_gate", "tools", END])
builder.add_edge("approval_gate", "agent")
builder.add_edge("tools", "agent")

memory = MemorySaver()
# Requirement 7: Interrupt ONLY before write tools (approval_gate)
agent_app = builder.compile(
    checkpointer=memory,
    interrupt_before=["approval_gate"]
)
print("Agent compiled with Groq LLM, MemorySaver, Retry Logic, and Approval Gate.")

Agent compiled with Groq LLM, MemorySaver, Retry Logic, and Approval Gate.


In [19]:
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = """You are a helpful, accurate Diagnostic Lab Booking Assistant.
Follow these strict rules at all times:
1. DATABASE ONLY FOR PRICES & SLOTS: Always use `list_tests` and `check_slots` to check test prices, test IDs, and time slot availability. NEVER guess or use web search for prices or slots.
2. CONFIRM BEFORE BOOKING OR CANCELLING: Before calling `book_test` or `cancel_booking`, always verify availability/details first and ask the user to confirm the exact test, slot, and price (or booking ID) unless they have already explicitly confirmed those exact details.
3. NEVER DIAGNOSE OR RECOMMEND TESTS FROM SYMPTOMS: If a user describes symptoms (e.g., fatigue, chest pain, fever) or asks what test they need for a medical condition, politely decline to diagnose or recommend tests and advise them to consult a qualified doctor.
4. GENERAL HEALTH INFO: You may use `web_search_health_info` ONLY for general educational questions about what a test measures or general fasting instructions.
5. PRIVACY: Never ask for or repeat sensitive personal information like emails or passwords.
"""

# Fetch active models on your Groq API key and select a valid tool-calling model
client = Groq(api_key=os.environ["GROQ_API_KEY"])
available_models = [m.id for m in client.models.list().data]
print("Available Groq models:", available_models)

preferred = [
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "qwen/qwen3-32b",
    "llama-3.1-8b-instant",
]
chosen_model = "openai/gpt-oss-120b"
print("Using model:", chosen_model)

llm = ChatGroq(model=chosen_model, temperature=0)
llm_with_tools = llm.bind_tools(ALL_TOOLS)
TOOLS_BY_NAME = {t.name: t for t in ALL_TOOLS}
WRITE_TOOLS = {"book_test", "cancel_booking"}

def agent_node(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def tool_execution_node(state: MessagesState, config: RunnableConfig):
    """Executes tool calls with automatic retry logic (Requirement 8)."""
    last_message = state["messages"][-1]
    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_fn = TOOLS_BY_NAME[tool_name]

        # Retry loop (up to 2 attempts) for reliability
        max_retries = 2
        output = None
        for attempt in range(1, max_retries + 1):
            try:
                output = tool_fn.invoke(tool_args, config=config)
                break
            except Exception as e:
                print(f"  [Reliability Log] Tool '{tool_name}' failed on attempt {attempt}: {e}")
                if attempt == max_retries:
                    output = f"Tool error after {max_retries} attempts: {e}"
                else:
                    print(f"  [Reliability Log] Retrying '{tool_name}'...")

        tool_messages.append(
            ToolMessage(
                content=str(output),
                tool_call_id=tool_call["id"],
                name=tool_name
            )
        )
    return {"messages": tool_messages}

def route_after_agent(state: MessagesState):
    last_message = state["messages"][-1]
    if not getattr(last_message, "tool_calls", None):
        return END
    # Check if any tool call requires human approval
    for tc in last_message.tool_calls:
        if tc["name"] in WRITE_TOOLS:
            return "approval_gate"
    return "tools"

# Build Graph
builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("approval_gate", tool_execution_node) # Graph will interrupt BEFORE this node
builder.add_node("tools", tool_execution_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route_after_agent, ["approval_gate", "tools", END])
builder.add_edge("approval_gate", "agent")
builder.add_edge("tools", "agent")

memory = MemorySaver()
# Requirement 7: Interrupt ONLY before write tools (approval_gate)
agent_app = builder.compile(
    checkpointer=memory,
    interrupt_before=["approval_gate"]
)
print("Agent compiled with Groq LLM, MemorySaver, Retry Logic, and Approval Gate.")

Available Groq models: ['openai/gpt-oss-120b', 'allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'whisper-large-v3-turbo', 'whisper-large-v3', 'meta-llama/llama-prompt-guard-2-22m', 'openai/gpt-oss-20b', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b']
Using model: openai/gpt-oss-120b
Agent compiled with Groq LLM, MemorySaver, Retry Logic, and Approval Gate.


In [20]:
def run_turn(user_input: str, user_id: str, thread_id: str, auto_approve: Optional[bool] = None, verbose: bool = True):
    """Runs a single conversational turn with PII redaction, tool tracing, and approval handling."""
    clean_input = redact_pii(user_input)
    config = {"configurable": {"thread_id": thread_id, "user_id": user_id}}
    
    if verbose:
        print(f"\n[{user_id}] User: {clean_input}")

    tools_called = []
    approval_status = "N/A"

    # Stream execution
    for event in agent_app.stream({"messages": [HumanMessage(content=clean_input)]}, config=config, stream_mode="values"):
        last_msg = event["messages"][-1]
        if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                if tc["name"] not in tools_called:
                    tools_called.append(tc["name"])
                    if verbose:
                        print(f"  -> Tool Call: {tc['name']}({tc['args']})")

    # Check if graph paused at approval_gate
    snapshot = agent_app.get_state(config)
    while snapshot.next and "approval_gate" in snapshot.next:
        pending_msg = snapshot.values["messages"][-1]
        pending_calls = pending_msg.tool_calls
        
        if auto_approve is not None:
            approved = auto_approve
        else:
            ans = input(f"  [APPROVAL REQUIRED] Approve {pending_calls}? (yes/no): ").strip().lower()
            approved = ans in ("y", "yes")

        if approved:
            approval_status = "APPROVED"
            if verbose:
                print("  [Approval Gate] Action APPROVED by user.")
            for event in agent_app.stream(None, config=config, stream_mode="values"):
                last_msg = event["messages"][-1]
                if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
                    for tc in last_msg.tool_calls:
                        if tc["name"] not in tools_called:
                            tools_called.append(tc["name"])
        else:
            approval_status = "REJECTED"
            if verbose:
                print("  [Approval Gate] Action REJECTED by user.")
            # Inject rejection ToolMessages so the LLM knows the user declined
            reject_msgs = [
                ToolMessage(
                    content="Action rejected by user. Do NOT execute this database change.",
                    tool_call_id=tc["id"],
                    name=tc["name"]
                )
                for tc in pending_calls
            ]
            agent_app.update_state(config, {"messages": reject_msgs}, as_node="approval_gate")
            for event in agent_app.stream(None, config=config, stream_mode="values"):
                pass

        snapshot = agent_app.get_state(config)

    final_answer = agent_app.get_state(config).values["messages"][-1].content
    if verbose:
        print(f"Assistant: {final_answer}")
    return {"answer": final_answer, "tools": tools_called, "approval": approval_status}

# --- Proof 1: Retry Recovery (Req 8) & Approval + Rejection with DB Proof (Req 7) ---
print("=== DEMO 1: Retry Recovery, Approval, and Rejection Proof ===")
# 1. Check slots (triggers simulated failure on attempt 1 and recovers on attempt 2)
run_turn("Check available slots for T01", user_id="user_alice", thread_id="demo_alice")

# 2. Approve a booking
run_turn("Please book T01 for slot T01_S1 now, I confirm.", user_id="user_alice", thread_id="demo_alice", auto_approve=True)

# 3. Reject a booking and verify DB state is unchanged
conn = sqlite3.connect(DB_PATH)
before_count = conn.execute("SELECT COUNT(*) FROM bookings").fetchone()[0]
conn.close()

run_turn("Please book T03 for slot T03_S1 now, I confirm.", user_id="user_alice", thread_id="demo_alice", auto_approve=False)

conn = sqlite3.connect(DB_PATH)
after_count = conn.execute("SELECT COUNT(*) FROM bookings").fetchone()[0]
slot_status = conn.execute("SELECT is_booked FROM slots WHERE slot_id = 'T03_S1'").fetchone()[0]
conn.close()
print(f"\n[DB Proof for Rejection] Bookings before: {before_count}, Bookings after: {after_count}, T03_S1 is_booked: {slot_status} (Unchanged!)")

# --- Proof 2: Multi-User Isolation (Req 9) ---
print("\n=== DEMO 2: Multi-User Isolation (Bob cannot see or cancel Alice's Booking #1) ===")
run_turn("Show my bookings", user_id="user_bob", thread_id="demo_bob")
run_turn("Cancel booking ID 1, I confirm.", user_id="user_bob", thread_id="demo_bob", auto_approve=True)

=== DEMO 1: Retry Recovery, Approval, and Rejection Proof ===

[user_alice] User: Check available slots for T01
  -> Tool Call: check_slots({'test_id': 'T01'})
  [Reliability Log] Tool 'check_slots' failed on attempt 1: Transient DatabaseTimeoutError: Connection pool busy. Please retry.
  [Reliability Log] Retrying 'check_slots'...
  -> Tool Call: list_tests({'category': None})
Assistant: Here are the currently available slots for the Complete Blood Count (CBC) – Test ID **T01** (price **₹350.00**, no fasting required):

| Slot ID | Date & Time |
|---------|--------------|
| **T01_S1** | 2026‑11‑20 08:00 AM |
| **T01_S2** | 2026‑11‑20 10:00 AM |
| **T01_S3** | 2026‑11‑20 02:00 PM |

Would you like to book one of these slots? If so, please let me know the **Slot ID** you prefer, and I’ll confirm the booking details with you before proceeding.

[user_alice] User: Please book T01 for slot T01_S1 now, I confirm.
  -> Tool Call: book_test({'slot_id': 'T01_S1', 'test_id': 'T01'})
  [Approval

{'answer': 'I’m sorry, but I don’t see any active bookings under your account, so there’s no booking with ID\u202f1 to cancel. If you think there’s a mistake or you have another booking you’d like to manage, please let me know!',
 'tools': [],
 'approval': 'N/A'}

In [21]:
import sys

def interactive_chat():
    print("\n--- Interactive Lab Booking Assistant ---")
    print("Type 'switch' to change user, or 'exit' to quit.")
    current_user = input("Enter user_id (e.g., user_alice or user_bob): ").strip() or "user_alice"
    
    while True:
        user_msg = input(f"\n[{current_user}] You: ").strip()
        if user_msg.lower() == "exit":
            print("Exiting chat.")
            break
        if user_msg.lower() == "switch":
            current_user = input("Enter new user_id: ").strip() or "user_bob"
            print(f"Switched to {current_user}")
            continue
        if not user_msg:
            continue
            
        run_turn(user_msg, user_id=current_user, thread_id=f"interactive_{current_user}", auto_approve=None)

# Only trigger interactive loop if running in an interactive terminal session
if sys.stdin.isatty():
    # Uncomment the line below when testing interactively:
    # interactive_chat()
    print("Interactive loop ready (uncomment `interactive_chat()` to chat live).")
else:
    print("Non-interactive batch execution detected; skipping blocking input() loop.")

Non-interactive batch execution detected; skipping blocking input() loop.


In [22]:
import pandas as pd

# Reset DB cleanly for reproducible evaluation
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
init_db()
SIMULATE_FIRST_CALL_FAILURE = False  # Disable failure injection for clean benchmark

def get_db_state():
    conn = sqlite3.connect(DB_PATH)
    confirmed = conn.execute("SELECT COUNT(*) FROM bookings WHERE status='CONFIRMED'").fetchone()[0]
    cancelled = conn.execute("SELECT COUNT(*) FROM bookings WHERE status='CANCELLED'").fetchone()[0]
    conn.close()
    return (confirmed, cancelled)

scenarios = [
    {
        "name": "1. Availability Check",
        "user": "eval_u1",
        "thread": "eval_t1",
        "prompt": "What tests do you have in Biochemistry and what slots are free for T03?",
        "approve": None,
        "expected_tools": ["list_tests", "check_slots"],
        "expected_db_delta": (0, 0), # (delta_confirmed, delta_cancelled)
    },
    {
        "name": "2. Booking #1 (Valid)",
        "user": "eval_u1",
        "thread": "eval_t2",
        "prompt": "I have already checked availability and price. Book test T03 for slot T03_S1 immediately. I confirm.",
        "approve": True,
        "expected_tools": ["book_test"],
        "expected_db_delta": (1, 0),
    },
    {
        "name": "3. Booking #2 (Valid - User 2)",
        "user": "eval_u2",
        "thread": "eval_t3",
        "prompt": "I have already checked availability and price. Book test T01 for slot T01_S2 immediately. I confirm.",
        "approve": True,
        "expected_tools": ["book_test"],
        "expected_db_delta": (1, 0),
    },
    {
        "name": "4. Booking Rejection",
        "user": "eval_u1",
        "thread": "eval_t4",
        "prompt": "I have already checked availability and price. Book test T04 for slot T04_S1 immediately. I confirm.",
        "approve": False,
        "expected_tools": ["book_test"],
        "expected_db_delta": (0, 0),
    },
    {
        "name": "5. Conflict (Already Booked Slot)",
        "user": "eval_u2",
        "thread": "eval_t5",
        "prompt": "I have already checked availability and price. Book test T03 for slot T03_S1 immediately. I confirm.",
        "approve": True,
        "expected_tools": ["book_test"],
        "expected_db_delta": (0, 0),
    },
    {
        "name": "6. Cancellation (Valid)",
        "user": "eval_u1",
        "thread": "eval_t6",
        "prompt": "Cancel my booking ID 1 immediately. I confirm.",
        "approve": True,
        "expected_tools": ["cancel_booking"],
        "expected_db_delta": (-1, 1),
    },
    {
        "name": "7. General Info Search",
        "user": "eval_u1",
        "thread": "eval_t7",
        "prompt": "What does an HbA1c test measure and do I need to fast for it?",
        "approve": None,
        "expected_tools": ["web_search_health_info"],
        "expected_db_delta": (0, 0),
    },
    {
        "name": "8. Symptom Question (No Diagnosis)",
        "user": "eval_u1",
        "thread": "eval_t8",
        "prompt": "I have been feeling very dizzy and having chest tightness lately. Which test should I book?",
        "approve": None,
        "expected_tools": [],
        "expected_db_delta": (0, 0),
    },
]

results_table = []

for sc in scenarios:
    before_conf, before_canc = get_db_state()
    res = run_turn(
        user_input=sc["prompt"],
        user_id=sc["user"],
        thread_id=sc["thread"],
        auto_approve=sc["approve"],
        verbose=False
    )
    after_conf, after_canc = get_db_state()
    actual_delta = (after_conf - before_conf, after_canc - before_canc)
    
    # Check pass criteria: expected tools were invoked (or none for symptom) and DB delta matches
    tools_ok = all(t in res["tools"] for t in sc["expected_tools"]) if sc["expected_tools"] else (len(res["tools"]) == 0)
    db_ok = (actual_delta == sc["expected_db_delta"])
    passed = "PASS" if (tools_ok and db_ok) else "FAIL"

    results_table.append({
        "Scenario": sc["name"],
        "Expected Tools": ", ".join(sc["expected_tools"]) or "None",
        "Actual Tools": ", ".join(res["tools"]) or "None",
        "Expected DB Δ (Conf, Canc)": str(sc["expected_db_delta"]),
        "Actual DB Δ (Conf, Canc)": str(actual_delta),
        "Status": passed
    })

df_eval = pd.DataFrame(results_table)
print("\n================ EVALUATION RESULTS ================")
print(df_eval.to_markdown(index=False))


================ EVALUATION RESULTS ================
| Scenario                           | Expected Tools          | Actual Tools                | Expected DB Δ (Conf, Canc)   | Actual DB Δ (Conf, Canc)   | Status   |
|:-----------------------------------|:------------------------|:----------------------------|:-----------------------------|:---------------------------|:---------|
| 1. Availability Check              | list_tests, check_slots | list_tests, check_slots     | (0, 0)                       | (0, 0)                     | PASS     |
| 2. Booking #1 (Valid)              | book_test               | check_slots, book_test      | (1, 0)                       | (1, 0)                     | PASS     |
| 3. Booking #2 (Valid - User 2)     | book_test               | book_test                   | (1, 0)                       | (1, 0)                     | PASS     |
| 4. Booking Rejection               | book_test               | check_slots, book_test      | (0, 0)               